# Quadmesh 500M — Google Colab (T4) training, start to end

Standalone notebook for the 500M tier only. Same setup as the 100M notebook - if you already ran that
one in this session/Drive, you can skip straight to section 7 here and reuse the same tokenizer.

**Precision: fp16 with gradient scaling, not bf16** - same reasoning as the 100M notebook.

**Before you start: Runtime -> Change runtime type -> T4 GPU.**

## 1. Confirm the GPU and precision Colab actually gave you

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else 'n/a')
# On a real T4 this should print: bf16 supported: False -> train_shared() will auto-pick fp16

## 2. Get the code into Colab

Upload `quadmesh_v6_generative.zip` to this Colab session's file browser (left sidebar -> folder icon
-> upload), then run this cell. Using Drive keeps checkpoints past a session reset - update the path if
you did not upload to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
ZIP_PATH = '/content/quadmesh_v6_generative.zip'   # or '/content/drive/MyDrive/quadmesh_v6_generative.zip'
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content')
os.chdir('/content/quadmesh_project')
print(os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!python tests/run_offline_tests.py

## 3. Storage that survives a session reset

Colab sessions are ephemeral - anything only in `/content` is lost if the runtime disconnects. Point
checkpoints at Drive so a disconnect does not cost you the whole run (`train_shared`'s `resume=True`
picks up from the last saved checkpoint either way).

In [ ]:
CKPT_DIR = '/content/drive/MyDrive/quadmesh_ckpts'
import os
os.makedirs(CKPT_DIR, exist_ok=True)
os.environ['CKPT_DIR'] = CKPT_DIR   # so `!` shell cells (e.g. training_report) can see it as $CKPT_DIR

## 4. Accounts

Text2CAD is a gated dataset - accept its license at
https://huggingface.co/datasets/SadilKhan/Text2CAD with the same account as your token, or the
tokenizer step below will show `(0 docs)` for it, same issue as on Modal.

In [ ]:
import getpass
from huggingface_hub import login
login(getpass.getpass('HF token: '))

## 5. Build the tokenizer (once - shared by both tiers)

In [ ]:
from quadmesh.data.tokenizer import get_or_train
tok = get_or_train(f'{CKPT_DIR}/tokenizer/quadmesh-bpe.json', vocab_size=32000)
print('vocab:', tok.vocab_size)

In [ ]:
# Imports needed by the training/eval cells below - these were missing, causing NameError
import shutil
from quadmesh.train.shared_text_train import train_shared
from quadmesh.agent.shared_runtime import get_runtime
from quadmesh.pipeline.shared_eval import evaluate

## 6. Review queue (optional - only needed for risk_tier_classifier/abuse_pattern)

Skip this whole section if you are only training the 6 generative roles, which is the reasonable
default given the budget constraint discussed earlier.

## 7. Train 500M - chinchilla budget

Same pattern, bigger tier. This will take meaningfully longer on a T4 - real time is not being
estimated here since you asked to optimize for $0 cost, not speed.

In [ ]:
result_500m_pretrain = train_shared(
    phase='pretrain', tier_name='500M', tok=tok,
    out_dir=f'{CKPT_DIR}/Quadmesh-500M/live',
    budget='chinchilla', device='cuda', precision=None, workers=2,
)
print(result_500m_pretrain)

In [ ]:
result_500m_sft = train_shared(
    phase='sft', tier_name='500M', tok=tok,
    out_dir=f'{CKPT_DIR}/Quadmesh-500M/live',
    tokens=2_000_000_000,
    init=f'{CKPT_DIR}/Quadmesh-500M/live/shared_pretrain.pt',
    device='cuda', precision=None, workers=2,
)
print(result_500m_sft)

In [ ]:
shutil.copy(f'{CKPT_DIR}/Quadmesh-500M/live/shared_sft.pt', f'{CKPT_DIR}/Quadmesh-500M/live/shared_text.pt')

In [ ]:
!python -m quadmesh.pipeline.training_report --tier 500M --ckpt-dir "$CKPT_DIR/Quadmesh-500M" --out /content/report_500m.md
!cat /content/report_500m.md

## 8. Report for the 500M run

In [ ]:
!python -m quadmesh.pipeline.training_report --tier 500M --ckpt-dir "$CKPT_DIR/Quadmesh-500M" --out /content/report_500m.md
!cat /content/report_500m.md

## 9. Eval

Uses the real `SharedRuntime`/`get_runtime` used by the agent and `serve.py` at inference time (not a
raw model call) - this is the same object `shared_eval.py`'s `evaluate()` expects as its `rt` argument,
so this matches exactly what a real Modal `shared_eval` run does.

In [ ]:
rt_500m = get_runtime(ckpt_root=CKPT_DIR, tier='500M', device='cuda')
assert rt_500m is not None, 'shared_text.pt not found for 500M - did the promotion cell above run?'
report_500m = evaluate(rt_500m, n_per_role=100)
print(report_500m)